In [5]:
import numpy as np
import pandas as pd
import os
import joblib

In [6]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
MODELS_DIR = "/home/mariya/Data/pallas/models"
OUTPUT_DIR = "/home/mariya/Development/pallas/reports/output/predictions"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_dataset(modality, split):
    dataset_path = os.path.join(DATASETS_PATH, f"{split}-{modality}")
    X_path = os.path.join(dataset_path, f"features.{modality}.csv")
    y_path = os.path.join(dataset_path, "labels.csv")

    X = pd.read_csv(X_path, index_col=0)
    y = pd.read_csv(y_path, index_col=0)

    return X, y


def load_train_dataset(modality):
    return load_dataset(modality, "train")


def load_test_dataset(modality, split):
    X, y = load_dataset(modality, split)
    y = y.iloc[:, 0].values  # Convert to 1D numpy array
    return X, y


def load_merged_test_datasets(modality):
    X_test, y_test = load_test_dataset(modality, "test")
    X_cal, y_cal = load_test_dataset(modality, "calibrate")

    # Append X_cal to X_test and y_cal to y_test
    X_merged = pd.concat([X_cal, X_test], axis=0)
    y_merged = np.concatenate([y_cal, y_test], axis=0)

    return X_merged, y_merged

def load_model(modality):
    model_path = os.path.join(MODELS_DIR, modality, f"final_model.pkl")
    return joblib.load(model_path)

In [7]:
def predict_for_modality_and_split(modality, split, model, X, y):
    print(f"Predicting for modality: {modality}, split: {split}")

    # Get predictions
    y_pred_proba = model.predict_proba(X)
    y_pred = model.predict(X)

    pred_df = pd.DataFrame(y_pred, columns=["predicted_label"], index=X.index)
    proba_df = pd.DataFrame(y_pred_proba, columns=model.classes_, index=X.index)

    # Make first column the true label, then predicted label, then probabilities
    proba_df.insert(0, "true_label", y)
    proba_df.insert(1, "predicted_label", y_pred)

    # Save predicted probabilities to CSV
    output_path = os.path.join(OUTPUT_DIR, f"{modality}_{split}_predictions.csv")
    proba_df.to_csv(output_path)
    print(f"Predicted probabilities saved to: {output_path}")

    return y_pred, y_pred_proba

def predict_for_modality(modality):
    model = load_model(modality)

    X_test, y_test = load_test_dataset(modality, "test")
    X_cal, y_cal = load_test_dataset(modality, "calibrate")
    X_merged, y_merged = load_merged_test_datasets(modality)

    predict_for_modality_and_split(modality, "test", model, X_test, y_test)
    predict_for_modality_and_split(modality, "calibrate", model, X_cal, y_cal)
    X_merged, y_merged = predict_for_modality_and_split(modality, "merged_test_calibrate", model, X_merged, y_merged)
    return X_merged, y_merged


X_gex, y_gex = predict_for_modality("gex")
X_dnam, y_dnam = predict_for_modality("dnam")

Predicting for modality: gex, split: test
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/predictions/gex_test_predictions.csv
Predicting for modality: gex, split: calibrate
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/predictions/gex_calibrate_predictions.csv
Predicting for modality: gex, split: merged_test_calibrate
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/predictions/gex_merged_test_calibrate_predictions.csv
Predicting for modality: dnam, split: test
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/predictions/dnam_test_predictions.csv
Predicting for modality: dnam, split: calibrate
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/predictions/dnam_calibrate_predictions.csv
Predicting for modality: dnam, split: merged_test_calibrate
Predicted probabilities saved to: /home/mariya/Development/pallas/reports/output/p